# 02 — Lambda0 / K_S0 / LambdaC purity optimization

**Stage 2 checkpoint notebook** for the BNV searches
$B^0 \to \Lambda^0 \Lambda^0$ and $B^+ \to \Lambda_c^+ \Lambda^0$.

Uses **MC only** (signal + luminosity-weighted background) -- no collision
data is read, per the Stage 2 plan (the optimization itself must not look
at data). This notebook:

1. re-optimizes the $\Lambda^0$ flight-significance cut and mass window
   with an $S/\sqrt{S+B}$ scan, separately for each channel;
2. (`Lam0LamC` only) does the same for $K_S^0$, **first** -- its cut gates
   two of the four $\Lambda_c^+$ decay modes, so it is fixed before the
   $\Lambda_c^+$ windows are set (sequential optimization order);
3. (`Lam0LamC` only) fits the per-mode $\Lambda_c^+$ mass resolution and
   derives `mass_window_nsigma`$\times\sigma$ windows per mode, with the
   $K_S^0$ gate applied, plus a before/after cross-check plot;
4. (`Lam0LamC` only) reports multi-candidate numbers after all Stage 2
   purity cuts -- **numbers only**, no candidate-selection policy yet.

This notebook does **not** modify `channel_config.py`. It prints the
recommended cut values for the human to review at the checkpoint; the same
computation is performed (and written to `results/<channel>.yaml`) by
`run_stage02.py`, which calls the same `purity_optimization` helpers this
notebook does, so the scan/fit logic cannot diverge between the two.

In [ ]:
import sys
sys.path.insert(0, '..')

%load_ext autoreload
%autoreload 2

import numpy as np
import awkward as ak
import pandas as pd
import matplotlib.pylab as plt

from channel_config import (get_channel_config, FOM_SIDEBAND_WIDTH_MULT,
                            BACKGROUND_SP_MODES, SIGNAL_SP_MODE)
import datasets
import cutflow
import purity_optimization as po
import plotting

In [ ]:
# Select the channel here: 'Lam0Lam0' or 'Lam0LamC'
CHANNEL = 'Lam0LamC'

config = get_channel_config(CHANNEL)
print(f"Channel: {config['name']}   {config['decay_label']}")

## Load signal + background MC only

No collision data is opened in this notebook (`sp_or_data='sp'`) -- the
purity-cut optimization must not look at data, per the Stage 2 plan.

In [ ]:
data_sp, _ = datasets.load_datasets(CHANNEL, sp_or_data='sp')
datasets.add_derived_fields(data_sp, config)

weights = datasets.get_scaling_weights(BACKGROUND_SP_MODES)

print(f"MC (SP) events: {len(data_sp)}")
print(f"Background scaling weights: {weights}")

## $\Lambda^0$ purity optimization (both channels)

Sequential 2-step $S/\sqrt{S+B}$ scan: (1) the flight-significance cut at
the *current* `channel_config.py` mass window, then (2) the mass
half-width around the PDG mass with that flight cut applied. $S$ is the
sideband-subtracted signal-MC yield in the mass-peak window; $B$ is the
sideband-estimated background from luminosity-weighted background MC (see
`purity_optimization.py` and `channel_config.FOM_SIDEBAND_WIDTH_MULT`).

In [ ]:
comp_l0 = config['composites']['Lambda0']

sig_l0 = po.flat_signal_arrays(data_sp, [comp_l0['mass_var'], comp_l0['flight_var']])
bkg_l0 = po.flat_background_arrays(data_sp, [comp_l0['mass_var'], comp_l0['flight_var']], weights)

print(f"Lambda0 candidates -- signal MC: {len(sig_l0[comp_l0['mass_var']])}, "
      f"background MC (weighted): {len(bkg_l0[comp_l0['mass_var']])}")

lambda0_opt = po.optimize_flight_and_mass(
    sig_l0, bkg_l0, comp_l0['mass_var'], comp_l0['flight_var'], comp_l0['mass_pdg'],
    comp_l0['mass_window'], comp_l0['flight_scan'], comp_l0['mass_halfwidth_scan'],
    FOM_SIDEBAND_WIDTH_MULT)

print(f"\nLambda0 flight cut:  current = {comp_l0['flight_cut']}, "
      f"recommended = {lambda0_opt['chosen_flight_cut']:.1f}  (FOM={lambda0_opt['fom_flight']:.2f})")
print(f"Lambda0 mass window: current = {comp_l0['mass_window']}, "
      f"recommended = {[round(x, 5) for x in lambda0_opt['chosen_mass_window']]}  "
      f"(FOM={lambda0_opt['fom_mass']:.2f})")

In [ ]:
plotting.plot_fom_scan(lambda0_opt['df_flight'], 'cut', chosen_x=lambda0_opt['chosen_flight_cut'],
                       xlabel=r'$\Lambda^0$ flight-significance cut',
                       title=f'{CHANNEL}: $\Lambda^0$ flight-significance scan')
plt.savefig(f"{plotting.plot_dir(config)}/lambda0_flight_scan.png", dpi=150)

chosen_hw_l0 = lambda0_opt['chosen_mass_window'][1] - comp_l0['mass_pdg']
plotting.plot_fom_scan(lambda0_opt['df_mass'], 'halfwidth', chosen_x=chosen_hw_l0,
                       xlabel=r'$\Lambda^0$ mass half-width [GeV]',
                       title=f'{CHANNEL}: $\Lambda^0$ mass-window scan (flight cut applied)')
plt.savefig(f"{plotting.plot_dir(config)}/lambda0_mass_scan.png", dpi=150)

In [ ]:
# Carry the Lambda0 recommendation forward; other composites are updated
# below (Lam0LamC only) before the multi-candidate study.
recommended_config = get_channel_config(CHANNEL)
recommended_config['composites']['Lambda0']['flight_cut'] = lambda0_opt['chosen_flight_cut']
recommended_config['composites']['Lambda0']['mass_window'] = lambda0_opt['chosen_mass_window']

## $K_S^0$ purity optimization (`Lam0LamC` only -- sequential, first)

Same 2-step scan as $\Lambda^0$. This must be fixed **before** the
$\Lambda_c^+$ per-mode mass windows below, since $K_S^0$ is a daughter of
the $\Lambda_c^+$ in decay modes 2 and 3 (`cutflow.get_lambdac_k0s_gate`)
and the LambdaC mass-resolution fit below is done with this gate applied.

In [ ]:
if CHANNEL == 'Lam0LamC':
    k0s = config['k0s']

    sig_k0s = po.flat_signal_arrays(data_sp, [k0s['mass_var'], k0s['flight_var']])
    bkg_k0s = po.flat_background_arrays(data_sp, [k0s['mass_var'], k0s['flight_var']], weights)

    print(f"K_S0 candidates -- signal MC: {len(sig_k0s[k0s['mass_var']])}, "
          f"background MC (weighted): {len(bkg_k0s[k0s['mass_var']])}")

    k0s_opt = po.optimize_flight_and_mass(
        sig_k0s, bkg_k0s, k0s['mass_var'], k0s['flight_var'], k0s['mass_pdg'],
        k0s['mass_window'], k0s['flight_scan'], k0s['mass_halfwidth_scan'],
        FOM_SIDEBAND_WIDTH_MULT)

    print(f"\nK_S0 flight cut:  current = {k0s['flight_cut']}, "
          f"recommended = {k0s_opt['chosen_flight_cut']:.1f}  (FOM={k0s_opt['fom_flight']:.2f})")
    print(f"K_S0 mass window: current = {k0s['mass_window']}, "
          f"recommended = {[round(x, 5) for x in k0s_opt['chosen_mass_window']]}  "
          f"(FOM={k0s_opt['fom_mass']:.2f})")

    plotting.plot_fom_scan(k0s_opt['df_flight'], 'cut', chosen_x=k0s_opt['chosen_flight_cut'],
                           xlabel=r'$K_S^0$ flight-significance cut',
                           title=f'{CHANNEL}: $K_S^0$ flight-significance scan')
    plt.savefig(f"{plotting.plot_dir(config)}/k0s_flight_scan.png", dpi=150)

    chosen_hw_k0s = k0s_opt['chosen_mass_window'][1] - k0s['mass_pdg']
    plotting.plot_fom_scan(k0s_opt['df_mass'], 'halfwidth', chosen_x=chosen_hw_k0s,
                           xlabel=r'$K_S^0$ mass half-width [GeV]',
                           title=f'{CHANNEL}: $K_S^0$ mass-window scan (flight cut applied)')
    plt.savefig(f"{plotting.plot_dir(config)}/k0s_mass_scan.png", dpi=150)

    recommended_config['k0s']['flight_cut'] = k0s_opt['chosen_flight_cut']
    recommended_config['k0s']['mass_window'] = k0s_opt['chosen_mass_window']

## $\Lambda_c^+$ per-mode mass resolution (`Lam0LamC` only)

Fit a Gaussian + linear background to each decay mode's $\Lambda_c^+$
candidate mass (signal MC, same histogram definition as the Stage 1
mode-split plot), with the recommended $K_S^0$ gate applied. The window
for each mode is `mass_window_nsigma`$\times\sigma_{\rm mode}$ around the
fitted mean -- **not** an $S/\sqrt{S+B}$ scan (the combinatorics here are
dominated by wrong-candidate combinations within signal MC itself, not by
a separate background sample, so the resolution fit is the more meaningful
quantity to cut on).

In [ ]:
if CHANNEL == 'Lam0LamC':
    mask_sig = data_sp['spmode'] == SIGNAL_SP_MODE

    mode = cutflow.get_lambdac_decay_mode(data_sp)
    mass_lamc = data_sp[config['composites']['LambdaC']['mass_var']]
    k0s_gate = cutflow.get_lambdac_k0s_gate(data_sp, recommended_config)

    mode_flat = ak.to_numpy(ak.flatten(mode[mask_sig], axis=None))
    mass_flat = ak.to_numpy(ak.flatten(mass_lamc[mask_sig], axis=None))
    gate_flat = ak.to_numpy(ak.flatten(k0s_gate[mask_sig], axis=None))

    lambdac_mass_pdg = config['composites']['LambdaC']['mass_pdg']
    nsigma = config['composites']['LambdaC']['mass_window_nsigma']

    mode_fits = {}
    recommended_windows = {}

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

    for ax, m in zip(axes, sorted(config['lambdac_modes'].keys())):
        sel_m = (mode_flat == m) & gate_flat
        fit = po.fit_mass_peak(mass_flat[sel_m], lambdac_mass_pdg)
        mode_fits[m] = fit
        recommended_windows[m] = po.nsigma_window(fit['mu'], fit['sigma'], nsigma)

        plotting.plot_mass_fit(fit, hist_def=config['hist_defs']['LambdaC_unc_Mass'],
                               title=f"mode {m}: {config['lambdac_modes'][m]}", ax=ax)

        print(f"mode {m}: mu={fit['mu']*1000:.2f} MeV  sigma={fit['sigma']*1000:.2f} MeV  "
              f"(converged={fit['converged']}, n={fit['n_candidates']})  "
              f"-> recommended window {[round(1000*x, 1) for x in recommended_windows[m]]} MeV")

    plt.tight_layout()
    plt.savefig(f"{plotting.plot_dir(config)}/lambdac_mode_mass_fits.png", dpi=150)

    recommended_config['composites']['LambdaC']['mass_windows_per_mode'] = recommended_windows

### Cross-check: LambdaC mass before/after the $K_S^0$ gate

Restricted to modes 2 and 3 (the only ones with a $K_S^0$ daughter -- the
gate is a no-op for modes 1 and 4). If the $\Lambda_c^+$ peak position and
width are essentially unchanged, the two optimizations are not strongly
coupled and the sequential order above is safe.

In [ ]:
if CHANNEL == 'Lam0LamC':
    mode_sig = mode[mask_sig]
    mass_sig_lamc = mass_lamc[mask_sig]
    gate_sig = k0s_gate[mask_sig]

    mask_ks_modes = (mode_sig == 2) | (mode_sig == 3)
    ks_mode_labels = {m: v for m, v in config['lambdac_modes'].items() if m in (2, 3)}

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    plotting.plot_split_by_mode(
        ak.flatten(mass_sig_lamc[mask_ks_modes]), ak.flatten(mode_sig[mask_ks_modes]),
        config['hist_defs']['LambdaC_unc_Mass'], ks_mode_labels,
        ax=axes[0], title=r'Before $K_S^0$ gate')

    mask_after = mask_ks_modes & gate_sig
    plotting.plot_split_by_mode(
        ak.flatten(mass_sig_lamc[mask_after]), ak.flatten(mode_sig[mask_after]),
        config['hist_defs']['LambdaC_unc_Mass'], ks_mode_labels,
        ax=axes[1], title=r'After $K_S^0$ gate')

    plt.tight_layout()
    plt.savefig(f"{plotting.plot_dir(config)}/lambdac_mass_before_after_k0s_gate.png", dpi=150)

## Multi-candidate study (`Lam0LamC` only)

After **all** Stage 2 purity cuts (recommended values above): the number
of $B$ candidates per signal-MC event whose composite daughters *both*
pass their purity masks (`cutflow.get_composite_purity_masks_per_B` -- a
per-$B$ check, stricter than the event-level "$n$ good $\Lambda^0$ == 1"
count used in Stage 1). **Numbers only** -- no candidate-selection policy
is applied at this stage; that decision is parked (see STATUS.md open
decisions).

In [ ]:
if CHANNEL == 'Lam0LamC':
    _, candidate_masks = cutflow.get_all_composite_purity_masks(data_sp, recommended_config)
    mask_b = cutflow.get_composite_purity_masks_per_B(data_sp, recommended_config, candidate_masks)

    n_good_b = ak.to_numpy(ak.sum(mask_b[mask_sig], axis=1))
    n_events = len(n_good_b)

    vals, counts = np.unique(n_good_b, return_counts=True)
    df_multi = pd.DataFrame({'n good B candidates': vals, 'n events': counts,
                             'fraction': counts / n_events})
    display(df_multi)

    print(f"\n{100*np.mean(n_good_b == 0):.1f}% of signal events: 0 good B candidates")
    print(f"{100*np.mean(n_good_b == 1):.1f}% of signal events: 1 good B candidate")
    print(f"{100*np.mean(n_good_b > 1):.1f}% of signal events: >1 good B candidates")

    plt.figure(figsize=(5, 3.5))
    plt.bar(vals, counts / n_events)
    plt.xlabel('# of good B candidates / event')
    plt.ylabel('fraction of signal events')
    plt.yscale('log')
    plt.savefig(f"{plotting.plot_dir(config)}/multi_candidate_study.png", dpi=150)

## Recommended `channel_config.py` updates

In [ ]:
print("Lambda0:")
print(f"  flight_cut  = {lambda0_opt['chosen_flight_cut']}")
print(f"  mass_window = {lambda0_opt['chosen_mass_window']}")

if CHANNEL == 'Lam0LamC':
    print("\nK_S0:")
    print(f"  flight_cut  = {k0s_opt['chosen_flight_cut']}")
    print(f"  mass_window = {k0s_opt['chosen_mass_window']}")
    print("\nLambdaC:")
    print(f"  mass_windows_per_mode = {recommended_windows}")
    print(f"  mass_window_nsigma    = {nsigma}  (unchanged)")

print("\nThese are NOT yet written into channel_config.py -- review at this "
      "checkpoint, then apply (Claude can do this on request) before Stage 3.")

## Observations / checkpoint summary

*(fill in after running -- items for the Stage 2 review)*

- [ ] Do the $\Lambda^0$ FOM scans show a clear, reasonably stable maximum
      (both channels), or a flat/noisy plateau that would need more MC
      statistics or a coarser grid to trust?

      They show a clear, stable maximum.

- [ ] Are the recommended $\Lambda^0$ cuts (flight significance, mass
      window) close to the p-Lambda0 defaults (25, $\pm 3$ MeV), or did
      they move a lot? If a lot, is there a reason specific to this
      channel?

     It's quite close at 18. 

- [ ] `Lam0LamC`: does the $K_S^0$ scan look sane (pre-fit mass resolution
      was estimated at ~7.8 MeV in Stage 1 -- does the recommended window
      match that order of magnitude)?

     The recommended window looks good. 

- [ ] `Lam0LamC`: did all four LambdaC mode fits converge? Do the fitted
      $\sigma$'s differ meaningfully between modes (motivating per-mode
      windows at all), and does `mass_window_nsigma = 3` look like a
      reasonable purity/efficiency trade-off, or should it change?

     They did all converge, though we might revisit this after the PID cuts.

- [ ] `Lam0LamC`: does the before/after $K_S^0$-gate cross-check plot show
      the LambdaC peak position/width essentially unchanged (validating
      the sequential K_S0-then-LambdaC optimization order)?

     Yes, they are effectively unchanged.


- [ ] `Lam0LamC`: multi-candidate numbers -- how much did the purity cuts
      reduce the >1-candidate fraction from the Stage 1 baseline? (No
      candidate-selection decision needed yet.)

      There is a much higher percentage of events with just 1 candidate. 
 

- [ ] Apply the recommended values to `channel_config.py` (ask Claude to
      do this once satisfied with the above), then re-run `run_stage01.py`
      and `run_stage02.py` to refresh `results/<channel>.yaml`, and
      `generate_latex_macros.py` to refresh the BAD.


    Will do. 

**Next steps:** review at this checkpoint, apply the chosen values to
`channel_config.py`, then Stage 3 (PID selector optimization).